# Galaxy Images from SKIRT

In this notebook you will look at a real TNG50 galaxy through the eyes of a virtual telescope.

Instead of building a galaxy model from scratch (as we explored conceptually in Notebook 3), we will use **pre-computed images** produced by a code called **SKIRT**.

---

## What is SKIRT?

**SKIRT** is a *radiative transfer* code. Rather than summing up stellar spectra mathematically, it simulates what actually happens to light inside a galaxy:

1. Millions of virtual **photon packets** are launched from the star particles
2. Each photon packet travels through the galaxy's gas and **dust**
3. Along the way, some photons are **absorbed** (especially in the UV), others are **scattered** in a new direction
4. The photons that escape are collected by a virtual "camera" and recorded in different filters

The result is much closer to what a real telescope would see than a simple spectrum sum — because it includes the full effect of **dust attenuation**.

SKIRT was run on TNG50 galaxies by a team at Ghent University, producing a public atlas of realistic galaxy images.

---

## What you will do

1. **Download** the SKIRT image package for a galaxy
2. **Explore** the available filter images (UV → mid-infrared)
3. **Understand** how the galaxy looks different at different wavelengths
4. **See the effect of dust** by comparing with and without it
5. **Make a colour image** by combining three filters
6. **Map the physical properties**: stellar mass, age, metallicity and dust

→ **Open `05_explore_discover.ipynb` afterwards** to go further with your own experiments.

In [ ]:
# Run this cell first — it installs everything you need.
# On COSMA or if packages are already installed this will be quick.

%pip install -q astropy requests
print("All packages ready!")

In [ ]:
# ============================================================
#  STEP 0 — SET YOUR API KEY AND CHOOSE A GALAXY
# ============================================================

API_KEY    = "693fc85106de8da5bf5a9eebf34c3fc5"   # ← your IllustrisTNG key
GALAXY_ID  = "553837"                              # ← the galaxy we will look at
ORIENT     = "O1"                                  # ← viewing angle (O1–O5)

# ============================================================
#  You do not need to change anything below this line.
# ============================================================

import sys, os
sys.path.insert(0, ".")
from helpers import (
    download_skirt_galaxy, extract_skirt_tar,
    skirt_file, load_fits_image,
    plot_skirt_image, plot_skirt_multiwave,
    plot_skirt_rgb, plot_skirt_property_maps,
    SKIRT_FILTERS, SKIRT_BASE_DIR,
)
import numpy as np
import matplotlib.pyplot as plt

print("Ready!")
print(f"Galaxy ID : {GALAXY_ID}")
print(f"Orientation: {ORIENT}  (O1 = face-on, O3 = edge-on, O5 = intermediate)")

---

## Step 1 — Download the SKIRT Image Package

The SKIRT atlas for each galaxy is bundled as a `.tar.gz` archive (~200 MB).
Once extracted you get **240 FITS image files** — one for every combination of:

| Dimension | Options |
|---|---|
| Viewing angle | O1, O2, O3, O4, O5 (5 different orientations) |
| Filter | 13 photometric bands from GALEX UV to WISE mid-IR, plus nodust variants |
| Property map | stellar mass, stellar age, metallicity, dust mass |

Each image is **1600 × 1600 pixels** at **100 pc per pixel**, so the total field of view is **160 kpc across**.

Run the cell below. It checks whether the data is already on disk before downloading.

In [ ]:
# Download and extract the SKIRT image package
tar_path, outdir = download_skirt_galaxy(GALAXY_ID, api_key=API_KEY)
extract_skirt_tar(tar_path, outdir)

print(f"\nData lives in: {outdir}")

---

## Step 2 — What Files Did We Get?

Let's look at what was extracted and understand the file naming convention.

In [ ]:
import glob

# List all FITS files for orientation O1
o1_files = sorted(glob.glob(f"{outdir}/TNG{int(GALAXY_ID):06d}_{ORIENT}_*.fits"))
print(f"FITS files for orientation {ORIENT}  ({len(o1_files)} total):\n")
for f in o1_files:
    print(" ", os.path.basename(f))

print()
print("File name format:  TNG{galaxy_id}_{orientation}_{instrument}_{filter}[_nodust].fits")
print()
print("Key:  no suffix  = image WITH dust  (what a real telescope sees)")
print("      _nodust    = image WITHOUT dust  (intrinsic starlight only)")

---

## Step 3 — Your First Galaxy Image

**FITS** (Flexible Image Transport System) is the standard file format in astronomy for storing images and data. Each FITS file here contains a 2D array of pixel values — just like a photograph, but in physical units (MJy/sr = megajansky per steradian, a measure of brightness per area of sky).

Let's load the LSST r-band image — the reddish optical filter used by the Vera Rubin Observatory.

In [ ]:
# Load the r-band image and display it
r_band = load_fits_image(skirt_file(GALAXY_ID, ORIENT, "LSST_r"))

print(f"Image size: {r_band.shape[0]} × {r_band.shape[1]} pixels")
print(f"Each pixel: 100 pc  →  total field of view: {r_band.shape[0] * 100 / 1000:.0f} kpc")
print(f"Brightness range: {r_band.min():.3e} – {r_band.max():.3e} MJy/sr")
print()

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0   # full field of view
XLIM    = None    # e.g. (-40, 40) to zoom in on the central 80 kpc
YLIM    = None
# ────────────────────────────────────────────────────────

plot_skirt_image(
    r_band,
    title  = f"Galaxy {GALAXY_ID} — LSST r-band (~6200 Å)",
    fov_kpc = FOV_KPC,
    xlim    = XLIM,
    ylim    = YLIM,
)

---

## Step 4 — Seeing the Same Galaxy at Different Wavelengths

A real galaxy looks completely different depending on which kind of light you observe it in:

| Wavelength region | What you see |
|---|---|
| **UV (GALEX FUV/NUV)** | Hot, young, massive stars — bright only where star formation is happening |
| **Optical (Johnson, LSST)** | A mix of all stars — shows the overall shape and structure |
| **Near-infrared (2MASS)** | Cool, old, low-mass stars — these dominate the total stellar mass |
| **Mid-infrared (WISE)** | Old stellar populations and warm dust emission |

The next cell shows galaxy `553837` across all 13 filters from UV to mid-IR.
Watch how the galaxy's appearance changes — the bright core, the arms, the overall size.

In [ ]:
# Show the galaxy in all filters from UV → mid-infrared
# SKIRT_FILTERS is a list of (filter_name, display_label, colour) tuples

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0
XLIM    = None    # e.g. (-50, 50)
YLIM    = None
# ────────────────────────────────────────────────────────

plot_skirt_multiwave(
    GALAXY_ID, ORIENT,
    filter_list = SKIRT_FILTERS,
    fov_kpc     = FOV_KPC,
    ncols       = 4,
    xlim        = XLIM,
    ylim        = YLIM,
)

print("Think about what you see:")
print("  - Which filter shows the most extended emission?")
print("  - Where is the galaxy brightest in the UV?  In the infrared?")
print("  - Can you see spiral structure in some filters but not others?")

---

## Step 5 — The Effect of Dust

Dust grains inside a galaxy absorb and scatter light — especially ultraviolet and blue light, which has a short wavelength similar in size to the dust grains. Longer-wavelength infrared light passes through much more easily.

The SKIRT atlas provides a **`_nodust` version** of every image: this is what the galaxy would look like if there were *no dust at all* — purely the intrinsic light from the stars.

By comparing the `_nodust` image with the real one, we can see exactly how much light the dust is hiding.

In [ ]:
# Compare three filters with and without dust, side by side
# (UV is most affected; infrared barely changes)

compare_filters = [
    ("GALEX_FUV", "GALEX FUV  (~1500 Å — far UV)"),
    ("Johnson_V",  "Johnson V  (~5500 Å — optical)"),
    ("2MASS_Ks",   "2MASS Ks   (~2.2 μm — near-IR)"),
]

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0
XLIM    = None
YLIM    = None
# ────────────────────────────────────────────────────────

from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, axes = plt.subplots(3, 3, figsize=(14, 14), facecolor="black")
half = FOV_KPC / 2

for row, (fname, flabel) in enumerate(compare_filters):
    with_dust  = load_fits_image(skirt_file(GALAXY_ID, ORIENT, fname))
    no_dust    = load_fits_image(skirt_file(GALAXY_ID, ORIENT, fname, nodust=True))
    # Ratio: nodust / with_dust  (>1 means dust is hiding light)
    ratio = no_dust / np.clip(with_dust, 1e-10, None)

    for col, (img, title, cmap) in enumerate([
        (with_dust,  f"{flabel}\nWith dust  (realistic)", "magma"),
        (no_dust,    f"{flabel}\nNo dust  (intrinsic)",   "magma"),
        (ratio,      f"{flabel}\nRatio: intrinsic / observed\n(>1 = dust is hiding light)",
                     "RdYlGn"),
    ]):
        ax = axes[row, col]
        if col < 2:
            arr = np.log10(np.clip(img, 1e-6, None))
        else:
            arr = np.log10(np.clip(img, 0.5, 100))  # log ratio
        im = ax.imshow(arr, origin="lower", cmap=cmap,
                       extent=[-half, half, -half, half])
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = fig.colorbar(im, cax=cax)
        cbar.ax.yaxis.set_tick_params(color="white")
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
        ax.set_title(title, color="white", fontsize=9, pad=4)
        ax.set_facecolor("black")
        ax.tick_params(colors="white", labelsize=7)
        ax.set_xlabel("x (kpc)", color="white", fontsize=7)
        ax.set_ylabel("y (kpc)", color="white", fontsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor("white")
        if XLIM: ax.set_xlim(XLIM)
        if YLIM: ax.set_ylim(YLIM)

fig.patch.set_facecolor("black")
fig.suptitle(f"Galaxy {GALAXY_ID} — Effect of Dust in Three Filters",
             color="white", fontsize=13)
plt.tight_layout()
plt.show()

print("Notice: the ratio is largest in the UV and smallest in the infrared.")
print("Dust absorbs short-wavelength light much more efficiently than long-wavelength light.")

---

## Step 6 — A Colour Image

A real telescope photograph is made by combining images taken through several filters and assigning each to a colour channel (red, green, blue). The colour you assign is a *choice* made by the astronomer — what matters is that redder filters go in the red channel.

Here we use:
- **Red channel** → LSST r-band (~6200 Å — red optical light)
- **Green channel** → LSST g-band (~4800 Å — green/blue optical light)
- **Blue channel** → Johnson U-band (~3650 Å — near-UV)

The colour of each region tells you something physically:
- **Blue areas** = lots of UV light = hot young stars = active star formation
- **Red/yellow areas** = old stellar population dominating = no recent star formation

In [ ]:
# Make an RGB colour image

# ── Filter choices ──────────────────────────────────────────────────────
R_FILTER = "LSST_r"    # ~6200 Å
G_FILTER = "LSST_g"    # ~4800 Å
B_FILTER = "Johnson_U" # ~3650 Å
STRETCH  = 0.05        # lower → brighter but less dynamic range

# ── Zoom controls ───────────────────────────────────────────────────────
FOV_KPC  = 160.0
XLIM     = None
YLIM     = None
# ────────────────────────────────────────────────────────────────────────

r_img = load_fits_image(skirt_file(GALAXY_ID, ORIENT, R_FILTER))
g_img = load_fits_image(skirt_file(GALAXY_ID, ORIENT, G_FILTER))
b_img = load_fits_image(skirt_file(GALAXY_ID, ORIENT, B_FILTER))

plot_skirt_rgb(
    r_img, g_img, b_img,
    title   = f"Galaxy {GALAXY_ID} — colour image  (R={R_FILTER}, G={G_FILTER}, B={B_FILTER})",
    stretch = STRETCH,
    fov_kpc = FOV_KPC,
    xlim    = XLIM,
    ylim    = YLIM,
)

print("Try changing R_FILTER / G_FILTER / B_FILTER to different bands")
print("and see how the image changes.")

---

## Step 7 — Physical Property Maps

SKIRT also outputs maps of the galaxy's physical properties at each pixel. These are not images of light — they are derived quantities calculated from the simulation particles that fall within each pixel:

| Map | What it shows | Units |
|---|---|---|
| **Stellar mass** | Total mass of stars in each pixel | M☉/pc² |
| **Stellar age** | Mass-weighted mean age of the stars | Gyr |
| **Metallicity** | Mass fraction of heavy elements in stars | (dimensionless) |
| **Dust mass** | Mass of dust grains per pixel | M☉/pc² |

Together, these tell the story of the galaxy: where the old stars live, where new stars are forming, and where the dust is hiding the light.

In [ ]:
# Display the four physical property maps

# ── Zoom controls ───────────────────────────────────────
FOV_KPC = 160.0
XLIM    = None
YLIM    = None
# ────────────────────────────────────────────────────────

plot_skirt_property_maps(
    GALAXY_ID, ORIENT,
    fov_kpc = FOV_KPC,
    xlim    = XLIM,
    ylim    = YLIM,
)

print("Questions to think about:")
print("  1. Does the stellar mass map look the same as the LSST r-band image?  Why might it differ?")
print("  2. Where are the youngest stars?  Where are the oldest?")
print("  3. Is the dust distributed the same way as the stellar mass?")
print("  4. How does the metallicity vary across the galaxy?")
print("     (Hint: metals are produced inside stars and released when they die —")
print("      older denser regions have had more time to build up metals.)")

---

## What You Have Done

You have just worked with real, research-grade data from the IllustrisTNG simulation, post-processed with full radiative transfer by the SKIRT code.

In summary:

```
TNG50 simulation                 SKIRT                        This notebook
────────────────                 ─────                        ─────────────
{star particle positions}        Trace photon packets         Filter images
{gas and dust distribution}  ──► through gas and dust    ──►  (with dust)
{star particle masses/ages}      Record escaping photons      +  nodust images
                                 in different filters         + property maps
```

This is exactly what real observers use when they want to compare simulations to telescope data — the TNG+SKIRT pipeline is used in published research papers.

---

**→ Open `05_explore_discover.ipynb` to start your own experiments!**